# Reproduce the paper's figures and table

This notebook regenerates **Fig. 2**, **Fig. 3** and **Table 1** of *ModalFidelity: Routing Modalities to Detect Deepfakes on a Budget* from the saved results in `data/`, then checks every number the paper states about them.

- **No training, no GPU, no downloads, no randomness.** Every value comes from `data/`, and `data/MANIFEST.json` records where each file came from.
- **Same plotting stack and style as the paper.** With the pinned `environment.yml`, the figures match the submitted ones pixel for pixel.
- **Run top to bottom.** It takes under a minute on a CPU.

In [ ]:
import os
from IPython.display import Image, Markdown, display

from modalfidelity.paths import REPO_ROOT, saved_data_dir
from modalfidelity.plots import claims, fig2, fig3, table1

DATA = saved_data_dir()
OUT = os.path.join(REPO_ROOT, "outputs", "figures")
# the figures are regenerated on every run, so replacing the previous run's copies is intended
OVERWRITE = True
print("data:", DATA)
print("figures ->", OUT)

## Fig. 2: with no budget cap, does picking the stream still matter?

Per-window confidence by forgery type. Confidence is the detector's margin $s_t^m-\tau^m$, rescaled per detector to $[-1,1]$, so 0 is the decision boundary. Every window is affordable, but at most one stream per window is read.

In [ ]:
png, pdf = fig2.plot(DATA, os.path.join(OUT, "fig2.png"), overwrite=OVERWRITE)
display(Image(png, width=640))

## Fig. 3: does spending more help without knowing where to look?

Accuracy (the mean of recall and specificity over windows) against the budget fraction $\rho$. Lines are the mean of four router seeds; the band is the range over seeds.

In [ ]:
png, pdf = fig3.plot(DATA, os.path.join(OUT, "fig3.png"), overwrite=OVERWRITE)
display(Image(png, width=640))

## Table 1: why route before the detectors rather than after?

Late MoE gates run both detectors on every window. The router decides before any detector runs.

In [ ]:
display(Markdown(table1.as_markdown(table1.load(DATA))))

## Every number the paper states about these results

Each row gives the paper's wording, the value recomputed from `data/`, and whether the paper's rounding holds.

In [ ]:
import pandas as pd

rows = claims.check(DATA)
table = pd.DataFrame(rows, columns=["claim", "paper", "recomputed", "match"])
display(table.style.format({"recomputed": "{:.4f}"}))
assert table["match"].all(), "a paper number was not reproduced"
print(f"{table['match'].sum()}/{len(table)} paper numbers reproduced")